--- 
## 1. Setup and Dependencies


In [1]:
# Install additional packages for fine-tuning
!pip install -q peft accelerate bitsandbytes evaluate seqeval biopython groq

import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoModel,
    AutoModelForTokenClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset, Dataset
import evaluate
import os
import requests
import ast
import re
import json
from sklearn.metrics import classification_report
from typing import List, Dict
from groq import Groq
import json


# Check for GPU availability for faster processing
device = "cuda" if torch.cuda.is_available() else "cpu"
# Force usage of only GPU 0. This hides the second GPU from Trainer.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 27.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 7.4 MB/s eta 0:00:00


2025-12-28 02:46:26.005028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766889986.271836      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766889986.350713      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766889986.998851      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766889986.998902      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766889986.998905      55 computation_placer.cc:177] computation placer alr

Using device: cpu
GPU: N/A


---
## 2. Loading Models and Data

Loading the Llama model for text generation and BioClinicalBERT for embeddings.

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
torch.cuda.is_available()

False

In [4]:
# Load NCBI disease dataset
db = pd.read_csv("/kaggle/input/datasetncbidisease/train.tsv", sep='\t')
db = db.dropna()
print(f"Dataset shape: {db.shape}")
print(f"Columns: {db.columns.tolist()}")
db.head()

Dataset shape: (135971, 2)
Columns: ['Identification', 'O']


,Identification,O
0,of,O
1,APC2,O
2,",",O
3,a,O
4,homologue,O


---
# PHASE 1: Zero-Shot DiRAG (Original Implementation)

This section implements your original zero-shot approach to establish a baseline.


## Setting Up PubMed API for RAG


In [5]:
from Bio import Entrez
import time
from urllib.error import HTTPError

# Configure PubMed API
user_secrets = UserSecretsClient()
MY_EMAIL = user_secrets.get_secret("email")
MY_API_KEY = user_secrets.get_secret("ncbi_token")

Entrez.email = MY_EMAIL
Entrez.api_key = MY_API_KEY

print("PubMed API configured!")

PubMed API configured!


In [6]:
def get_context(search_term, search_db="pubmed"):
    """This function extracts the documents which are required for the context for Zero-Shot DiRAG module"""
    try:
        handle = Entrez.esearch(db=search_db, term=search_term, retmax=5)
        record = Entrez.read(handle)
        handle.close()
        return record["IdList"]
    except Exception as e:
        print(f"Search error for {search_term}: {e}")
        return []

print("Context retrieval function ready!")

Context retrieval function ready!


## Zero-Shot Entity Identification Workflow

### Step 1: Identification of Potential Entities

In [7]:
# Creating a database of Punctuations and stopwords to be removed for consideration for predictions
import string
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"]
punctuation_list = list(string.punctuation)
punctuation_list.extend(stopwords)

print(f"Stopwords and punctuation list created: {len(punctuation_list)} items")

Stopwords and punctuation list created: 159 items


In [8]:
import pandas as pd
import math

# Sample size
SAMPLE_SIZE = 500
# Get the raw list of words
raw_words = db["Identification"].iloc[0:SAMPLE_SIZE].tolist()

# 1. PRE-PROCESSING: Filter punctuation first, then chunk
clean_words = [w for w in raw_words if w not in punctuation_list]

# Define Chunk Size (Sentence length)
CHUNK_SIZE = 30
# Create lists of 20 words: [['word1', 'word2'...], ['word21'...]]
chunks = [clean_words[i:i + CHUNK_SIZE] for i in range(0, len(clean_words), CHUNK_SIZE)]

all_prompts = []

system_instruction = """You are a biomedical expert. 
Task: You will receive a list of words. Classify EACH word in the list sequentially.

Rules:
1. You must respond with a valid JSON Object where keys are the words and values are classification characters.
2. 'e' = Clinical and scientific words
3. 'o' = General words
4. Use ONLY 'e' or 'o' as values.
5. Provide ONLY the JSON object, no extra text or explanations.

Example Input: [diabetes, is, bad]
Example Output: {"diabetes": "e", "is": "o", "bad": "o"}
"""

# 3. CREATE PROMPTS
for chunk in chunks:
    # Convert list of words to a string representation for the prompt
    chunk_str = str(chunk) 
    
    prompt_content = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Word List: {chunk_str}\n\nClassifications:"}
    ]
    all_prompts.append(prompt_content)

print(f"Created {len(all_prompts)} prompts (batches) for {len(clean_words)} words.")
print("Running inference...")


Created 10 prompts (batches) for 289 words.
Running inference...


In [9]:
# # 1. Initialize the Kaggle Secrets client
user_secrets = UserSecretsClient()

# 2. Retrieve the secret value using the label you created
secret_key = user_secrets.get_secret("groq_api")

client = Groq(api_key=secret_key)

def process_with_groq(messages):
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile", 
            # FIX: 'messages' should be the list itself, not wrapped in another list
            messages=messages, 
            temperature=0, 
            max_tokens=2000,
            # Force JSON mode to ensure the output is a valid dictionary
            response_format={"type": "json_object"} 
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Example usage with your prompts_final list
final_outputs = []
current_batch = 1
for p in all_prompts:
    result = process_with_groq(p)
    final_outputs.append(result)
    print(f"Processed batch... {current_batch}")
    current_batch+= 1

print("All results received!")



Processed batch... 1
Processed batch... 2
Processed batch... 3
Processed batch... 4
Processed batch... 5
Processed batch... 6
Processed batch... 7
Processed batch... 8
Processed batch... 9
Processed batch... 10
All results received!


In [10]:
final_outputs

['{\n  "APC2": "e",\n   "homologue": "e",\n   "adenomatous": "e",\n   "polyposis": "e",\n   "coli": "e",\n   "tumour": "e",\n   "suppressor": "e",\n   "The": "o",\n   "adenomatous": "e",\n   "polyposis": "e",\n   "coli": "e",\n   "APC": "e",\n   "tumour": "e",\n   "suppressor": "e",\n   "protein": "e",\n   "controls": "o",\n   "Wnt": "e",\n   "signalling": "e",\n   "pathway": "e",\n   "forming": "o",\n   "complex": "e",\n   "glycogen": "e",\n   "synthase": "e",\n   "kinase": "e",\n   "3beta": "e",\n   "GSK": "e",\n   "3beta": "e",\n   "axin": "e",\n   "conductin": "e",\n   "betacatenin": "e"\n}',
 '{\n  "Complex": "o",\n   "formation": "e",\n   "induces": "e",\n   "rapid": "o",\n   "degradation": "e",\n   "betacatenin": "e",\n   "In": "o",\n   "colon": "e",\n   "carcinoma": "e",\n   "cells": "e",\n   "loss": "o",\n   "APC": "e",\n   "leads": "o",\n   "accumulation": "e",\n   "betacatenin": "e",\n   "nucleus": "e",\n   "binds": "e",\n   "activates": "e",\n   "Tcf": "e",\n   "4": "o",\n 

In [11]:
import json

# 5. POST-PROCESSING (Unpacking JSON results)
final_results = {}

# Zip your original chunks with the raw string outputs from Groq
for chunk_words, output_string in zip(chunks, final_outputs):
    try:
        # Step A: Parse the string into a real Python dictionary
        # output_string is something like: '{"APC2": "e", "homologue": "o"}'
        batch_dict = json.loads(output_string)
        
        # Step B: Map labels to our global final_results dictionary
        for word in chunk_words:
            # We use .get() to handle cases where the model might miss a word
            # Defaulting to 'o' (Not a disease)
            label = batch_dict.get(word, 'o').lower()
            
            # Final normalization: ensure we store 'e' or 'o'
            final_results[word] = 'e' if 'e' in label else 'o'
            
    except Exception as e:
        print(f"Error parsing batch: {e}")
        # If a whole batch fails, mark all words in that chunk as 'o'
        for word in chunk_words:
            final_results[word] = 'o'

print("Classification complete!")
print(f"Sample results: {list(final_results.items())[:5]}")

Classification complete!
Sample results: [('APC2', 'e'), ('homologue', 'e'), ('adenomatous', 'e'), ('polyposis', 'e'), ('coli', 'e')]


In [12]:
# 1. Create the DataFrame
db_temp = pd.DataFrame(list(final_results.items()), columns=["word", "prediction"])

# 2. Filter using Pandas logic (prediction is 'e' AND word not in punctuation)
filtered_df = db_temp[
    (db_temp["prediction"] == "e") & 
    (~db_temp["word"].isin(punctuation_list))
]

# 3. Get the list
words_for_rag = filtered_df["word"].tolist()

print(f"Found {len(words_for_rag)} potential entity words.")
print(words_for_rag[:10])

Found 91 potential entity words.
['APC2', 'homologue', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', 'APC', 'protein', 'Wnt']


In [13]:
words_for_rag

['APC2',
 'homologue',
 'adenomatous',
 'polyposis',
 'coli',
 'tumour',
 'suppressor',
 'APC',
 'protein',
 'Wnt',
 'signalling',
 'pathway',
 'complex',
 'glycogen',
 'synthase',
 'kinase',
 '3beta',
 'GSK',
 'axin',
 'conductin',
 'betacatenin',
 'formation',
 'induces',
 'degradation',
 'colon',
 'carcinoma',
 'cells',
 'accumulation',
 'nucleus',
 'binds',
 'activates',
 'Tcf',
 'transcription',
 'factor',
 'identification',
 'genomic',
 'structure',
 'homologues',
 'Mammalian',
 'domain',
 'SAMP',
 'domains',
 'binding',
 'regulates',
 'complexes',
 'transient',
 'transcriptional',
 'activation',
 'assays',
 'Human',
 'chromosome',
 '19p13',
 'functions',
 'development',
 'cancer',
 'MSH2',
 'mutation',
 'HNPCC',
 'phenotypic',
 'expression',
 'colorectal',
 'frequency',
 'germline',
 'gene',
 'kindreds',
 'hereditary',
 'syndrome',
 'T',
 'nt943',
 'splice',
 'exon',
 'deletion',
 'mRNA',
 'analysed',
 'extensive',
 'analysis',
 'reduced',
 'investigate',
 'haplotype',
 'microsa

### Step 2: RAG-Based Entity Identification

In [14]:
def get_context_xml(words_list, max_tokens=300):
    """Retrieve PubMed context for multiple words combined"""
    if not words_list or len(words_list) == 0:
        return ""
    
    # Combine words with AND for multi-word search
    search_term = " AND ".join(words_list) + "[TIAB]"
    
    ids = get_context(search_term, "pubmed")
    
    # If no results with AND, try OR for broader search
    if not ids and len(words_list) > 1:
        search_term = " OR ".join(words_list) + "[TIAB]"
        ids = get_context(search_term, "pubmed")
    
    valid_ids = [str(j) for j in ids if j]
    
    if valid_ids:
        try:
            time.sleep(0.3)  # Rate limiting
            list_of_ids = ",".join(valid_ids[:5])
            handle = Entrez.efetch(db="pubmed", id=list_of_ids, retmode="xml")
            context_xml = handle.read()
            handle.close()
            
            if isinstance(context_xml, bytes):
                text = context_xml.decode('utf-8', errors='ignore')
            else:
                text = context_xml
                
            clean_text = re.sub(r'<[^>]+>', ' ', text)
            clean_text = re.sub(r'\s+', ' ', clean_text).strip()
            tokens = llama_tokenizer.encode(clean_text)
            
            if len(tokens) > max_tokens:
                tokens = tokens[:max_tokens]
            clean_text = llama_tokenizer.decode(tokens, skip_special_tokens=True)
            
            return clean_text
            
        except HTTPError as e:
            print(f"HTTP Error: {e}")
            return ""
        except Exception as e:
            print(f"Error: {e}")
            return ""
    else:
        return ""

print("Context retrieval function ready!")

Context retrieval function ready!


In [15]:
system_instruction = """You are a biomedical NER expert specializing in disease entity recognition.
Task: You will receive a list of words. Classify EACH word sequentially based on the provided context.

Rules:
- You must respond with a valid JSON Object where keys are the words and values are classification characters.
- 'Disease' = Disease (e.g., Cancer, tumor, coli, diabetes)
- 'O' = Not a disease (e.g., anatomy, procedures, medications, symptoms, general terms)
- Output ONLY a valid Python dictionary with words as keys and 'Disease' or 'O' as values.
- Do not include any other text, explanations, or the context in your response.

Example Output: {"coli": "e", "english": "o"}
"""
all_prompts = []
def create_final_prompts(word_list, batch_size):
# 3. CREATE PROMPTS
   for i in range(0, len(word_list),batch_size):
        words_for_context = word_list[i:i+batch_size]
        context = get_context_xml(words_for_context)
        
        # Convert list of words to a string representation
        chunk_str = str(words_for_context) 
        
        prompt_content = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": f"Word List: {chunk_str}\n Context: {context}"}
        ]
        all_prompts.append(prompt_content)
       
   print(f"Created {len(all_prompts)} prompts for {len(clean_words)} words.")
   return all_prompts 
   print("Example prompt structure ready for inference.")

In [16]:
# Create final prompts with context
prompts_final = create_final_prompts(words_for_rag,batch_size=10)
print(f"Created {len(prompts_final)} final prompts")

Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Error: name 'llama_tokenizer' is not defined
Created 10 prompts for 289 words.
Created 10 final prompts


In [17]:
# 1. Initialize the Kaggle Secrets client
user_secrets = UserSecretsClient()

# 2. Retrieve the secret value using the label you created
secret_key = user_secrets.get_secret("groq_api")

client = Groq(api_key=secret_key)

def process_with_groq(messages):
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile", 
            # FIX: 'messages' should be the list itself, not wrapped in another list
            messages=messages, 
            temperature=0, 
            max_tokens=1000,
            # Force JSON mode to ensure the output is a valid dictionary
            response_format={"type": "json_object"} 
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Example usage with your prompts_final list
final_outputs = []
batch_count = 1
for p in prompts_final:
    result = process_with_groq(p)
    final_outputs.append(result)
    print(f"Processed batch... {batch_count}")
    batch_count+= 1

print("All results received!")

Processed batch... 1
Processed batch... 2
Processed batch... 3
Processed batch... 4
Processed batch... 5
Processed batch... 6
Processed batch... 7
Processed batch... 8
Processed batch... 9
Processed batch... 10
All results received!


In [18]:
final_outputs

['{\n  "APC2": "O",\n   "homologue": "O",\n   "adenomatous": "O",\n   "polyposis": "Disease",\n   "coli": "Disease",\n   "tumour": "Disease",\n   "suppressor": "O",\n   "APC": "O",\n   "protein": "O",\n   "Wnt": "O"\n}',
 '{\n  "signalling": "O",\n   "pathway": "O",\n   "complex": "O",\n   "glycogen": "O",\n   "synthase": "O",\n   "kinase": "O",\n   "3beta": "O",\n   "GSK": "O",\n   "axin": "O",\n   "conductin": "O"\n}',
 '{\n  "betacatenin": "O",\n   "formation": "O",\n   "induces": "O",\n   "degradation": "O",\n   "colon": "O",\n   "carcinoma": "Disease",\n   "cells": "O",\n   "accumulation": "O",\n   "nucleus": "O",\n   "binds": "O"\n}',
 '{\n  "activates": "O",\n   "Tcf": "O",\n   "transcription": "O",\n   "factor": "O",\n   "identification": "O",\n   "genomic": "O",\n   "structure": "O",\n   "homologues": "O",\n   "Mammalian": "O",\n   "domain": "O"\n}',
 '{\n  "SAMP": "O",\n   "domains": "O",\n   "binding": "O",\n   "regulates": "O",\n   "complexes": "O",\n   "transient": "O",\n 

In [19]:
import pandas as pd
import ast

# 1. PARSING: Convert strings to real dictionaries and flatten them into one list
all_rows = []

for item in final_outputs:
    try:
        # Convert the string representation of the dict to a real dict
        # We use .strip() to remove any accidental whitespace/newlines from the API
        real_dict = ast.literal_eval(item.strip())
        
        # Flatten the dictionary: {'word': 'classification'} -> [{'word': 'w', 'classification': 'c'}]
        for word, label in real_dict.items():
            all_rows.append({
                "word": word,
                "classification": label
            })
    except Exception as e:
        print(f"Skipping a malformed batch: {e}")

# 2. DATAFRAME CREATION
db_predicted = pd.DataFrame(all_rows)

# 3. CLEANUP: Map 'e' to 'Disease' and 'o' to 'O' (matching your requirement)
db_predicted['classification'] = db_predicted['classification'].replace({'e': 'Disease', 'o': 'O'})


In [20]:
# Map words to predictions
prediction_map = dict(zip(db_predicted["word"], db_predicted["classification"]))

# Get predictions for all SAMPLE_SIZE words
y_pred_raw = db["Identification"].iloc[0:SAMPLE_SIZE].map(prediction_map).fillna("O")

# Normalize predictions: 'e'/'o' → 'Disease'/'O'
y_pred = y_pred_raw.apply(lambda x: "Disease" if x in ['e', 'Disease'] else "O").tolist()

# Normalize ground truth: B-Disease/I-Disease → 'Disease', O → 'O'
y_true = db["O"].iloc[0:SAMPLE_SIZE].apply(
    lambda x: "Disease" if x.startswith("B-") or x.startswith("I-") else "O"
).tolist()

# Calculate metrics
from sklearn.metrics import classification_report, f1_score

print("="*70)
print("ZERO-SHOT RESULTS")
print("="*70)
print(classification_report(y_true, y_pred, labels=["Disease", "O"]))
print(f"\nF1 Score (Disease): {f1_score(y_true, y_pred, pos_label='Disease'):.3f}")
print("="*70)

# Show prediction summary
print(f"\nTotal predictions: {len(y_pred)}")
print(f"Predicted Disease: {y_pred.count('Disease')}")
print(f"Predicted O: {y_pred.count('O')}")
print(f"True Disease: {y_true.count('Disease')}")
print(f"True O: {y_true.count('O')}")

ZERO-SHOT RESULTS
              precision    recall  f1-score   support

     Disease       1.00      0.68      0.81        40
           O       0.97      1.00      0.99       460

    accuracy                           0.97       500
   macro avg       0.99      0.84      0.90       500
weighted avg       0.97      0.97      0.97       500


F1 Score (Disease): 0.806

Total predictions: 500
Predicted Disease: 27
Predicted O: 473
True Disease: 40
True O: 460
